In [1]:
"""
Detector: сферический детектор с N фотоумножителями.

Параметры по умолчанию:
    R = 2.0 м   — радиус поверхности детектора
    N = 40      — число фотоумножителей
    r = 0.10 м  — радиус одного фотоумножителя (10 см)
    a = 0.5     — коэффициент подъёма при визуализации (чтобы ФЭУ были видны над сферой)

Расположение ФЭУ — по спирали Фибоначчи (равномерно по сфере).
"""

import numpy as np
import k3d


# ---------- вспомогательные функции ----------

def _uv_sphere(R, n_u=72, n_v=36):
    """Сетка UV-сферы: возвращает (vertices, indices)."""
    u = np.linspace(0.0, 2.0 * np.pi, n_u, endpoint=False)
    v = np.linspace(0.0, np.pi, n_v)
    U, V = np.meshgrid(u, v)
    X = R * np.sin(V) * np.cos(U)
    Y = R * np.sin(V) * np.sin(U)
    Z = R * np.cos(V)
    verts = np.stack([X, Y, Z], axis=-1).reshape(-1, 3).astype(np.float32)

    idx = []
    for i in range(n_v - 1):
        for j in range(n_u):
            j2 = (j + 1) % n_u
            a = i * n_u + j
            b = i * n_u + j2
            c = (i + 1) * n_u + j
            d = (i + 1) * n_u + j2
            idx.append([a, c, b])
            idx.append([b, c, d])
    return verts, np.asarray(idx, dtype=np.uint32)


def _cap_mesh(direction, R, theta, n=24):
    """
    Сферическая «шапочка» радиуса R с угловым полураствором theta
    вокруг направления direction. Возвращает (vertices, indices).
    """
    d = direction / np.linalg.norm(direction)
    tmp = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(d, tmp)) > 0.9:
        tmp = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(d, tmp); e1 /= np.linalg.norm(e1)
    e2 = np.cross(d, e1)

    verts = [R * d]
    for phi in np.linspace(0.0, 2.0 * np.pi, n, endpoint=False):
        p = (R * np.cos(theta) * d
             + R * np.sin(theta) * (np.cos(phi) * e1 + np.sin(phi) * e2))
        verts.append(p)
    verts = np.asarray(verts, dtype=np.float32)

    idx = []
    for i in range(1, n + 1):
        j = i + 1 if i < n else 1
        idx.append([0, i, j])
    return verts, np.asarray(idx, dtype=np.uint32)


# ---------- основной класс ----------

class Detector:
    def __init__(self, R=2.0, N=40, r=0.10, a=0.5):
        self.R = float(R)
        self.N = int(N)
        self.r = float(r)
        self.a = float(a)

        # Угловой радиус одной «шапочки» ФЭУ, видимой из центра сферы
        self.theta_pmt = float(np.arcsin(self.r / self.R))

        # Равномерное расположение ФЭУ (спираль Фибоначчи)
        self.pmt_directions = self._fibonacci_sphere(self.N)      # (N, 3) — единичные векторы
        self.pmt_centers = self.pmt_directions * self.R            # (N, 3) — центры на сфере

        # Структура для хранения информации о каждом ФЭУ
        self.pmts = [
            {
                "id": i,
                "direction": self.pmt_directions[i],
                "center": self.pmt_centers[i],
                "radius": self.r,
                "theta": self.theta_pmt,
            }
            for i in range(self.N)
        ]

    # ---- генерация точек ----
    @staticmethod
    def _fibonacci_sphere(N):
        idx = np.arange(N, dtype=float)
        phi = np.pi * (3.0 - np.sqrt(5.0))     # золотой угол
        y = 1.0 - 2.0 * (idx + 0.5) / N
        r_xy = np.sqrt(np.clip(1.0 - y * y, 0.0, 1.0))
        theta = phi * idx
        x = np.cos(theta) * r_xy
        z = np.sin(theta) * r_xy
        return np.stack([x, y, z], axis=1)

    # ---- геометрическая проверка ----
    def belongs_to_pmt(self, point):
        """
        Принадлежит ли точка (x, y, z) поверхности какому-либо ФЭУ?
        Возвращает id ФЭУ или None.
        """
        p = np.asarray(point, dtype=float)
        norm = np.linalg.norm(p)
        if norm == 0.0:
            return None
        p_dir = p / norm
        cos_ang = self.pmt_directions @ p_dir                # (N,)
        angles = np.arccos(np.clip(cos_ang, -1.0, 1.0))
        k = int(np.argmin(angles))
        return k if angles[k] <= self.theta_pmt else None

    # ---- визуализация ----
    def build_plot(self, sphere_color=0x3050a0, pmt_color=0xffcc00):
        plot = k3d.plot(name="Detector")

        # 1) Поверхность детектора (полупрозрачная сфера)
        verts, indices = _uv_sphere(self.R)
        plot += k3d.mesh(verts, indices,
                         color=sphere_color, opacity=0.15,
                         wireframe=False)

        # 2) Фотоумножители — единый меш из сферических «шапочек»
        all_v, all_i = [], []
        offset = 0
        for pmt in self.pmts:
            v, i = _cap_mesh(pmt["direction"],
                             self.R + self.a * self.r,   # слегка приподняты для наглядности
                             self.theta_pmt,
                             n=24)
            all_v.append(v)
            all_i.append(i + offset)
            offset += len(v)

        plot += k3d.mesh(np.vstack(all_v), np.vstack(all_i),
                         color=pmt_color, wireframe=False)
        return plot


# ---------- демонстрация ----------

if __name__ == "__main__":
    det = Detector(R=2.0, N=40, r=0.10, a=0.5)
    print(f"Число ФЭУ: {len(det.pmts)}")
    print(f"Угловой радиус ФЭУ: {det.theta_pmt:.5f} рад")

    # Точка точно в центре первого ФЭУ
    p1 = det.pmt_centers[0]
    print("Точка в центре ФЭУ №0 ->", det.belongs_to_pmt(p1))

    # Точка рядом с центром ФЭУ №0 (смещение 3 см по нормали)
    p2 = det.pmt_centers[0] * 1.015
    print("Точка рядом с ФЭУ №0 ->", det.belongs_to_pmt(p2))

    # Случайная точка между ФЭУ
    rng = np.random.default_rng(0)
    rand = rng.normal(size=3); rand /= np.linalg.norm(rand)
    p3 = rand * det.R
    print("Случайная точка ->", det.belongs_to_pmt(p3))

    # Визуализация (в Jupyter — просто `plot`; в скрипте — `plot.display()`)
    plot = det.build_plot()
    plot.display()

Число ФЭУ: 40
Угловой радиус ФЭУ: 0.05002 рад
Точка в центре ФЭУ №0 -> 0
Точка рядом с ФЭУ №0 -> 0
Случайная точка -> None


Output()